# Jev by Example: Choice, Noul & Score — and a Jev-Gated Agent Loop

**Jev** (TypeSafe AI) is a *discriminative* model which can help make decision. You hand it a **state** (text or JSON) plus one or more **typed questions**, and it returns **typed answers with probabilities** without any additional text or explaination. Your code acts on those answers directly.

This notebook has two parts:

1. **Using Jev for different types of usage** — Test code for Choice, Noul (yes/no) and Score, each on its own.
2. **Strands agent with Jev as the checker tool** — a Strands agent whose *quit thinking loop criteria* is decided by a Jev-based tool.

**Prerequisites:** an OpenRouter API key with credit, and (for Part 2) a SageMaker execution role with Amazon Bedrock model access, since Strands uses Claude on Bedrock by default.

## Part 0 - Setup

## Accessing Jev

Jev is reachable through several routes. You don't have to call TypeSafe directly.

| Option | What you need | Best when |
|--------|---------------|-----------|
| **OpenRouter** *(used here)* | One OpenRouter API key | Quickest start — no separate TypeSafe account, no waitlist |
| **TypeSafe SDK** (`typesafe-sdk`) | A TypeSafe account + key | You want typed Python objects instead of raw JSON |
| **TypeSafe HTTP API** | A TypeSafe account + key | Plain REST, no SDK |
| **Cloudflare / DigitalOcean / Vercel** | An account on that platform | You already run on one of them |

**This notebook uses OpenRouter** 

#### Creating an OpenRouter API key

**Sign in** at [openrouter.ai](https://openrouter.ai) using Google, GitHub, or email to create API Key and add credit for the consumption.

**Keep the key private.** Don't commit it or paste it into shared notebooks. A spend limit on the key caps the damage if it ever leaks, and you can revoke a key anytime from the Keys page.

In [ ]:
#!pip install -q strands-agents requests

In [ ]:
import os
from getpass import getpass
import requests

if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("Enter OpenRouter API key: ")

OPENROUTER_API_KEY = os.environ["OPENROUTER_API_KEY"]

## Part 1 — Using Jev for different types of usage

### Usage 1: Using Jev for choice selection

In [ ]:
def choice_call(message: str) -> dict:
    r = requests.post(
        "https://openrouter.ai/api/alpha/decisions",
        headers={"Authorization": f"Bearer {OPENROUTER_API_KEY}"},
        json={
            "model": "typesafe/jev-1.13",
            "state": {"message": message},
            "questions": {
                "team": {
                    "type": "choice",
                    "instructions": "Which team should handle `message`?",
                    "criteria": {"payments": "Charges, refunds", "fraud": "Stolen cards", "lending": "Loans"},
                }
            },
        },
        timeout=10,
    )
    r.raise_for_status()
    return r.json()["answers"]["team"]

message = "Someone used my card in another country"   # change me
answer = choice_call(message)
print(answer["choice"], answer["confidence"], answer["probabilities"])

### Usage 2: Using Jev for yes/no binary answers

In [ ]:
def has_pii_data(message: str) -> float:
    r = requests.post(
        "https://openrouter.ai/api/alpha/decisions",
        headers={"Authorization": f"Bearer {OPENROUTER_API_KEY}"},
        json={
            "model": "typesafe/jev-1.13",
            "state": {"message": message},
            "questions": {
                "has_pii": {
                    "type": "noul",
                    "instructions": "Does `message` contain personal data (name, card number, address)?",
                }
            },
        },
        timeout=10,
    )
    r.raise_for_status()
    return r.json()["answers"]["has_pii"]["noul"]

p = has_pii_data("My name is Anna Meyer and my card is 4111 1111 1111 1111.")
print(f"P(cofidence of containing PII data) = {p:.2f}")

### Usage 3: Using Jev for scoring

In [ ]:
def sentiment(message: str) -> dict:
    r = requests.post(
        "https://openrouter.ai/api/alpha/decisions",
        headers={"Authorization": f"Bearer {OPENROUTER_API_KEY}"},
        json={
            "model": "typesafe/jev-1.13",
            "state": {"message": message},
            "questions": {
                "sentiment": {
                    "type": "score",
                    "instructions": "How does the author of `message` feel about our service?",
                    "criteria": [
                        "Very unhappy",
                        "Unhappy",
                        "Neutral",
                        "Happy",
                        "Very happy",
                    ],
                }
            },
        },
        timeout=10,
    )
    r.raise_for_status()
    return r.json()["answers"]["sentiment"]

s = sentiment("Honestly the new app is fantastic — fastest support I've ever had!")
print(f"score {s['score']:.2f} on a 0-4 scale")   # e.g. 3.85
# 0 = Very unhappy, 1 = Unhappy, 2 = Neutral, 3 = Happy, 4 = Very happy

## Part 2 — Strands agent with Jev as the checker tool
We are writing a content draft agent. Agent uses `write_draft` tool to create content. Agent also uses  `draft_complete_check` tool which decides DONE / NOT DONE check on content draft to let agent know if it should stop drafting any further. The check tool using Jev model. There is no loop cap for the agent.

In [ ]:
from strands import Agent, tool

# Deliberately start rough so there is something to improve.
writer = Agent(
    system_prompt=(
        "You write concise business content. On your FIRST attempt, produce a quick, "
        "rough draft only. Improve it on each later attempt using the feedback given."
    ),
    callback_handler=None,
    model="us.anthropic.claude-haiku-4-5-20251001-v1:0"
)

@tool
def write_draft(task: str, feedback: str = "") -> str:
    """Write or improve a draft for the task.

    Args:
        task: What needs to be produced.
        feedback: What the checker said is still wrong (empty on first try).
    """
    prompt = f"Task: {task}"
    if feedback:
        prompt += f"\nThe previous draft was rejected because: {feedback}. Fix exactly that."
    return str(writer(prompt))

@tool
def draft_complete_check(task: str, draft: str) -> str:
    """Check whether a draft fully completes the task. Always call after write_draft.

    Args:
        task: The original task.
        draft: The output returned by write_draft.
    """
    r = requests.post(
        "https://openrouter.ai/api/alpha/decisions",
        headers={"Authorization": f"Bearer {OPENROUTER_API_KEY}"},
        json={
            "model": "typesafe/jev-1.13",
            "state": {"task": task, "draft": draft},
            "questions": {
                # Several independent bars. ALL must pass -> early drafts rarely clear them.
                "meets_all": {
                    "type": "noul",
                    "instructions": (
                        "Does `draft` satisfy EVERY requirement in `task` exactly: the precise "
                        "count of items, the required format, a specific concrete example in each "
                        "item, and no filler? Answer true only if all hold."
                    ),
                },
                "issue": {
                    "type": "choice",
                    "instructions": "What is the single biggest problem with `draft` for `task`?",
                    "criteria": {
                        "none": "No real problem at all",
                        "too_vague": "Points are generic and lack concrete, specific examples",
                        "wrong_count_or_format": "Wrong number of items or wrong format/length",
                        "incomplete": "Parts of the task are missing",
                        "off_topic": "Content does not address the task",
                    },
                },
            },
        },
        timeout=10,
    )
    r.raise_for_status()
    a = r.json()["answers"]
    p_done, issue = a["meets_all"]["noul"], a["issue"]["choice"]
    print(f"[check] p_done={p_done:.2f} issue={issue}")

    # Higher bar than before (0.5), so a rough first draft won't pass.
    if p_done >= 0.5 and issue == "none":
        return f"DONE (p={p_done:.2f}). Stop and return the draft."
    return f"NOT DONE (p={p_done:.2f}, issue={issue}). Call write_draft again, fixing: {issue}."

agent = Agent(
    tools=[write_draft, draft_complete_check],
    system_prompt=(
        # Tool name matches the real function, so the loop is reliable.
        "You improve a draft in rounds. Steps: (1) call write_draft. "
        "(2) call draft_complete_check on its output. (3) If it says NOT DONE, call write_draft "
        "again, passing the stated issue as feedback, then check again. Repeat until "
        "draft_complete_check says DONE, then return the final draft. Do not stop early."
    ),
     model="us.anthropic.claude-haiku-4-5-20251001-v1:0"
)

# A demanding task gives the loop something to work on.
result = agent(
    "Write exactly 5 bullet points on why banks should adopt GenAI for customer service. "
    "Each bullet must name a specific, concrete example (a named use case or metric) and be "
    "under 20 words."
)
print("\n=== FINAL ===\n", result)